## PyQtGraph Guide: Building a Temperature & Precipitation Chart

This notebook contains a step-by-step breakdown creating on a dual-axis temperature and precipitation chart using PyQtGraph, starting from an empty `PlotWidget` and building up to a full weather visualization. PyQtGraph is a plotting library built on PyQt. This makes it tightly integrated into the PyQT Framework, requiring it to be run inside a PyQT application. This disadvantage, however, also yields a few advantages compared to, for example, matplotlib:
- It makes use of Qt's GraphicsView framework, which in turn uses GPU acceleration when available. This allows it to render constantly changing real-time data smoothly and without slowing the rest of the application down.
- It features rich interactive features, such as zooming and panning, auto-range button, and a context menu enabling many visualization options, such as hiding/showing grid lines

The Weather app does not make use of these advantages (data is real time, but only updated every 10 Minutes), and interactivity is disabled. This means that for the weather app, selecting matplotlib in the settings does not lead to any disadvantages. I still decided to make PyQtgraph the default selection for the app and to create the guide for it instead of for matplotlib, due to wanting to learn its usage for other projects which will profit from its speed.

### Prepwork 1
We'll need to enable the Qt eventloop inside the notebook

In [1]:
%gui qt

### Prepwork 2
- import needed libraries
- create sample data
- Implement a basic application to hold the PyQtGraph

In [2]:
import sys
import numpy as np
import pandas as pd
import pyqtgraph as pg
from PyQt6.QtWidgets import QApplication, QMainWindow, QVBoxLayout, QWidget
from PyQt6.QtCore import Qt

# Create Sample data
temperature = [
    6, 5, 5, 4, 4, 5, 7, 9, 12, 15, 18, 21,
    23, 24, 24, 23, 21, 18, 15, 12, 10, 8, 7, 6
]

feels_like = [
    5, 4, 4, 3, 3, 4, 6, 8, 11, 14, 17, 20,
    22, 23, 23, 22, 20, 17, 14, 11, 9, 7, 6, 5
]

precipitation = [
    0, 0, 0, 0, 0.2, 0.3, 0.5, 0.8, 0.4, 0.2, 0,
    0, 0, 0, 0, 0.1, 0.3, 0.6, 1.2, 0.8, 0.4, 0.2, 0, 0
]

data = pd.DataFrame(
    {
        "temperature": temperature,
        "feels_like": feels_like,
        "precipitation": precipitation,
    }
)

# Create the application
app = QApplication.instance()
if app is None:
    app = QApplication(sys.argv)


# Create the main window of the application
class WeatherApp(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setGeometry(100, 100, 800, 600)
        central_widget = QWidget()
        self.setCentralWidget(central_widget)
        layout = QVBoxLayout(central_widget)
        self.plot_widget = create_weather_chart(data)
        layout.addWidget(self.plot_widget)

### Step 1: Creating a PyqtGraph
We'll start as simple as possible, by just returning an empty plot.

In [3]:
def create_weather_chart(data) -> pg.PlotWidget:
    """Create an empty plot widget"""
    widget = pg.PlotWidget()
    return widget


# Create and show the window
# NOTE: This will open a seperate window
window = WeatherApp()
window.show()

### Step 2: Plotting data
Temperature is a rather important weather information. We can start with this, creating a simple line plot that displays the temperature

In [4]:
def create_weather_chart(data) -> pg.PlotWidget:
    """Create a plot widget plotting a single temperature line"""
    widget = pg.PlotWidget()
    widget.setMinimumHeight(300)
    widget.setBackground("w")  # White background
    widget.showGrid(x=True, y=True, alpha=0.3)  # Light grid
    widget.setTitle("Temperature Over 24 Hours", color="k", size="14pt")  # Title
    widget.setLabel(
        "left", "Temperature", units="°C", color="#e74c3c"
    )  # Temperature label on the left y-axis
    widget.setLabel("bottom", "Hour")  # Hour label on the bottom x-axis
    temperature = data["temperature"]
    hours = np.arange(len(data))

    pen = pg.mkPen(color="#e74c3c", width=3)  # configure the lines styling
    widget.plot(hours, temperature, pen=pen, name="Temperature")  # plot the data

    return widget


window = WeatherApp()
window.show()

### Step 3: Adding more data
The temperature we feel is not only influenced by the real temperature, but also for example by wind. As such we are also interested in the feels-like temperature.

In [5]:
def create_weather_chart(data) -> pg.PlotWidget:
    """Create a plot widget plotting two temperature lines"""
    widget = pg.PlotWidget()
    widget.setMinimumHeight(300)
    widget.setBackground("w")
    widget.showGrid(x=True, y=True, alpha=0.3)
    widget.setTitle("Temperature & Feels Like", color="k", size="14pt")
    widget.setLabel("left", "Temperature", units="°C", color="#e74c3c")
    widget.setLabel("bottom", "Hour")
    temperature = data["temperature"]
    feels_like = data["feels_like"]
    hours = np.arange(len(data))

    # add a legend to tell which line displays what
    widget.addLegend()  # PyQtGraph collects every item that is added after this function call (so we add the legend BEFORE adding the lines)

    temp_pen = pg.mkPen(color="#e74c3c", width=3)
    widget.plot(hours, temperature, pen=temp_pen, name="Temperature")

    # Plotting additional lines is straightforward
    # Use a different color and style to tell the lines apart
    feels_pen = pg.mkPen(color="#ff9800", width=3, style=Qt.PenStyle.DashLine)
    widget.plot(hours, feels_like, pen=feels_pen, name="Feels Like")

    return widget


window = WeatherApp()
window.show()

### Step 4: Creating a dual axis plot
We want to have more information, than just the temperature. For example precicipation would be nice. For this we create a second ViewBox (the first ViewBox is created indirectly as part of the PlotWidget).

In [6]:
def create_second_viewbox(widget: pg.PlotWidget) -> pg.ViewBox:
    """Creates a viewbox and links it to the given widget"""
    precip_viewbox = pg.ViewBox()
    widget.scene().addItem(precip_viewbox)
    # Link the secondary ViewBox to the axis that it uses (the right y-axis that belongs to it, and the shared bottom x-axis)
    widget.getAxis("right").linkToView(precip_viewbox)
    precip_viewbox.setXLink(widget)

    # Define update function to sync ViewBoxes
    def update_views():
        precip_viewbox.setGeometry(widget.getViewBox().sceneBoundingRect())

    precip_viewbox.linkedViewChanged(widget.getViewBox(), precip_viewbox.XAxis)
    update_views()
    # Connect resize signal
    widget.getViewBox().sigResized.connect(update_views)
    return precip_viewbox


def create_weather_chart(data) -> pg.PlotWidget:
    """Create a plot widget plotting two temperature lines and a precicipation line"""
    widget = pg.PlotWidget()
    widget.setMinimumHeight(300)
    widget.setBackground("w")
    widget.showGrid(x=True, y=True, alpha=0.3)
    widget.setTitle("Temperature & Precipitation", color="k", size="14pt")
    widget.setLabel("left", "Temperature", units="°C", color="#e74c3c")
    widget.setLabel("bottom", "Hour")
    widget.showAxis("right")  # Enable the right y-axis
    widget.setLabel("right", "Precipitation", units="mm", color="#3498db")
    legend = widget.addLegend()
    temperature = data["temperature"]
    feels_like = data["feels_like"]
    precipitation = data[
        "precipitation"
    ].values  # tiny detail: widget.plot() can handle pd.Series, but PlotCurveItem needs the data to be np.ndarray
    hours = np.arange(len(data))

    temp_pen = pg.mkPen(color="#e74c3c", width=3)
    widget.plot(hours, temperature, pen=temp_pen, name="Temperature")
    feels_pen = pg.mkPen(color="#ff9800", width=3, style=Qt.PenStyle.DashLine)
    widget.plot(hours, feels_like, pen=feels_pen, name="Feels Like")

    # I would like the precipitation to be plotted as a bar chart using the second (right) y-axis
    # This isn't straightforward. One way to do it is by adding another viewbox
    # Here I'll plot the precicipation as a line to introduce complexity more gradually changing to a bar chart in the next step
    precip_viewbox = create_second_viewbox(widget)

    # Add precipitation line to the secondary ViewBox
    precip_pen = pg.mkPen(color="#3498db", width=2, style=Qt.PenStyle.DotLine)
    # ViewBox does not have a plot method. Instead we plot the line, then add it to the ViewBox
    precip_curve = pg.PlotCurveItem(hours, precipitation, pen=precip_pen)
    precip_viewbox.addItem(precip_curve)
    # Note that the precip line is not added to the legend automatically (due to not being a direct item of the widget)
    legend.addItem(precip_curve, "Precipitation")
    return widget


window = WeatherApp()
window.show()

### Step 5: Introducing a new Chart type (Bar chart) to the plot
Line Plots aren't well suited to display the per-hour summed up precicipation. We'll use a Bar chart instead

In [7]:
def create_weather_chart(data) -> pg.PlotWidget:
    """Create a plot widget plotting two temperature lines and precicipation bars"""
    widget = pg.PlotWidget()
    widget.setMinimumHeight(300)
    widget.setBackground("w")
    widget.showGrid(x=True, y=True, alpha=0.3)
    widget.setTitle("Temperature & Precipitation", color="k", size="14pt")
    widget.setLabel("left", "Temperature", units="°C", color="#e74c3c")
    widget.setLabel("bottom", "Hour")
    widget.showAxis("right")  # Enable the right y-axis
    widget.setLabel("right", "Precipitation", units="mm", color="#3498db")
    widget.addLegend()
    temperature = data["temperature"]
    feels_like = data["feels_like"]
    precipitation = data["precipitation"].values
    hours = np.arange(len(data))

    temp_pen = pg.mkPen(color="#e74c3c", width=3)
    widget.plot(hours, temperature, pen=temp_pen, name="Temperature")
    feels_pen = pg.mkPen(color="#ff9800", width=3, style=Qt.PenStyle.DashLine)
    widget.plot(hours, feels_like, pen=feels_pen, name="Feels Like")

    # Adding a bar chart to the second viewbox is straightforward
    precip_viewbox = create_second_viewbox(widget)
    precip_pen = pg.mkPen(color="#3498db", width=2)
    brush = "#3498db80"  # the prefix #3498db defines the color, the suffix 80 defines the opacity
    bar_graph = pg.BarGraphItem(
        x=hours, height=precipitation, width=0.6, brush=brush, pen=precip_pen
    )
    precip_viewbox.addItem(bar_graph)

    return widget


window = WeatherApp()
window.show()

### Step 6: Reworking the horizontal grid-lines
You might have noticed that the two y-axis do not align. This leads to the horizontal grid lines of the temperature plot not being at the same positions as the horizontal grid lines for the precicipation chart. This can be fixed by manually setting both axis.

In [55]:
def get_nice_axis_range(data_min, data_max):
    """Calculate a nice range with round tick intervals."""
    data_range = data_max - data_min
    # Determine a nice interval based on range
    if data_range <= 2:
        interval = 0.5
    elif data_range <= 5:
        interval = 1
    elif data_range <= 10:
        interval = 2
    elif data_range <= 25:
        interval = 5
    else:
        interval = 10
    # Calculate nice min and max
    nice_min = np.floor(data_min / interval) * interval
    nice_max = np.ceil(data_max / interval) * interval
    # Ensure a minimum range of 1
    if nice_max - nice_min < 1:
        nice_max = nice_min + 1
    return nice_min, nice_max, interval


def create_weather_chart(data) -> pg.PlotWidget:
    """Create a plot widget plotting two temperature lines and precicipation bars"""
    widget = pg.PlotWidget()
    widget.setMinimumHeight(300)
    widget.setBackground("w")
    widget.showGrid(x=True, y=True, alpha=0.3)
    widget.setTitle("Temperature & Precipitation", color="k", size="14pt")
    widget.setLabel("left", "Temperature", units="°C", color="#e74c3c")
    widget.setLabel("bottom", "Hour")
    widget.showAxis("right")  # Enable the right y-axis
    widget.setLabel("right", "Precipitation", units="mm", color="#3498db")
    widget.addLegend()
    temperature = data["temperature"]
    feels_like = data["feels_like"]
    precipitation = data["precipitation"].values
    hours = np.arange(len(data))

    temp_pen = pg.mkPen(color="#e74c3c", width=3)
    widget.plot(hours, temperature, pen=temp_pen, name="Temperature")
    feels_pen = pg.mkPen(color="#ff9800", width=3, style=Qt.PenStyle.DashLine)
    widget.plot(hours, feels_like, pen=feels_pen, name="Feels Like")
    precip_viewbox = create_second_viewbox(widget)
    precip_pen = pg.mkPen(color="#3498db", width=2)
    brush = "#3498db80"
    bar_graph = pg.BarGraphItem(
        x=hours, height=precipitation, width=0.6, brush=brush, pen=precip_pen
    )
    precip_viewbox.addItem(bar_graph)

    # Calculate aligned axis ranges
    temp_min = min(temperature.min(), feels_like.min())
    temp_max = max(temperature.max(), feels_like.max())
    precip_min = 0
    precip_max = precipitation.max()
    # Get nice ranges for both axes
    temp_nice_min, temp_nice_max, temp_interval = get_nice_axis_range(
        temp_min, temp_max
    )
    precip_nice_min, precip_nice_max, precip_interval = get_nice_axis_range(
        precip_min, precip_max
    )
    # Calculate the number of ticks for each axis
    temp_num_ticks = int((temp_nice_max - temp_nice_min) / temp_interval) + 1
    precip_num_ticks = int((precip_nice_max - precip_nice_min) / precip_interval) + 1
    # Extend the axis with fewer ticks to match the one with more ticks
    target_num_ticks = max(temp_num_ticks, precip_num_ticks)
    temp_nice_max = temp_nice_min + temp_interval * (target_num_ticks - 1)
    precip_nice_max = precip_nice_min + precip_interval * (target_num_ticks - 1)
    # Set the y-range of the first (left) y-axis
    widget.setYRange(temp_nice_min, temp_nice_max)
    # Set the y-range of the second (right) y-axis
    precip_viewbox.setYRange(precip_nice_min, precip_nice_max)

    return widget


window = WeatherApp()
window.show()

### Step 7: Disabling Interactivity
We can easily destroy the hard work of setting up the y-axis by zooming in/out of the chart or by using the auto-range button. 
We don't really need interactivity (there's not many data points that zooming in/out is needed), so we disable it

In [57]:
def create_weather_chart(data) -> pg.PlotWidget:
    """Create a plot widget plotting two temperature lines and precicipation bars"""
    widget = pg.PlotWidget()
    widget.setMinimumHeight(300)
    widget.setBackground("w")
    widget.showGrid(x=True, y=True, alpha=0.3)
    widget.setTitle("Temperature & Precipitation", color="k", size="14pt")
    widget.setLabel("left", "Temperature", units="°C", color="#e74c3c")
    widget.setLabel("bottom", "Hour")
    widget.showAxis("right")  # Enable the right y-axis
    widget.setLabel("right", "Precipitation", units="mm", color="#3498db")
    widget.addLegend()

    # Disable interactivity. Remember to also do this for the second ViewBox
    widget.setMouseEnabled(x=False, y=False)
    widget.setMenuEnabled(False)
    widget.hideButtons()

    temperature = data["temperature"]
    feels_like = data["feels_like"]
    precipitation = data["precipitation"].values
    hours = np.arange(len(data))

    temp_pen = pg.mkPen(color="#e74c3c", width=3)
    widget.plot(hours, temperature, pen=temp_pen, name="Temperature")
    feels_pen = pg.mkPen(color="#ff9800", width=3, style=Qt.PenStyle.DashLine)
    widget.plot(hours, feels_like, pen=feels_pen, name="Feels Like")
    precip_viewbox = create_second_viewbox(widget)
    precip_pen = pg.mkPen(color="#3498db", width=2)
    brush = "#3498db80"
    bar_graph = pg.BarGraphItem(
        x=hours, height=precipitation, width=0.6, brush=brush, pen=precip_pen
    )
    precip_viewbox.addItem(bar_graph)

    temp_min = min(temperature.min(), feels_like.min())
    temp_max = max(temperature.max(), feels_like.max())
    precip_min = 0
    precip_max = precipitation.max()
    temp_nice_min, temp_nice_max, temp_interval = get_nice_axis_range(
        temp_min, temp_max
    )
    precip_nice_min, precip_nice_max, precip_interval = get_nice_axis_range(
        precip_min, precip_max
    )
    temp_num_ticks = int((temp_nice_max - temp_nice_min) / temp_interval) + 1
    precip_num_ticks = int((precip_nice_max - precip_nice_min) / precip_interval) + 1
    target_num_ticks = max(temp_num_ticks, precip_num_ticks)
    temp_nice_max = temp_nice_min + temp_interval * (target_num_ticks - 1)
    precip_nice_max = precip_nice_min + precip_interval * (target_num_ticks - 1)
    widget.setYRange(temp_nice_min, temp_nice_max)
    precip_viewbox.setYRange(precip_nice_min, precip_nice_max)

    # Disable interactivity of the second ViewBox
    precip_viewbox.setMouseEnabled(x=False, y=False)
    precip_viewbox.setMenuEnabled(False)

    return widget


window = WeatherApp()
window.show()

## Summary
1. **PlotWidget Start**: Our starting point was an empty PlotWidget
2. **Adding data**: We can use `plot()` to draw a simple line plot in the widget
3. **Adding more data** By repeatedly calling `plot()` we can draw multiple line plots in a single widget. We can use `pg.mkPen()` to different between the different series of data
4. **Viewboxes** We can use `pg.ViewBox()` to create a second ViewBox to hold precicipation data
5. **Bar Chart** PyQtGraph supports many plot types, not just line plots. We can use `pg.BarGraphItem()` to create a Bar chart which we can then add using `addItem()`
6. **Reworking grid lines** Manually setting the left and right y-axis allows us to align them, ensuring that the horizontal grid lines of both axes are at the same positions
7. **Disabling interactivity** PyQtGraphs are interactive by default, but this can be easily disabled if needed